# Part 1: Ready Made vs Custom Made Data

# Part 2: Find Researchers using the OpenAlex API

In [11]:
import pandas as pd
from dotenv import load_dotenv
import requests
import os
from datetime import datetime
import backoff # https://pypi.org/project/backoff/
from tqdm.contrib.concurrent import thread_map
import json
from collections import Counter # For taking the mode of a list
import Levenshtein as L # Distance metric
from rapidfuzz import fuzz # For tokenizing words
from itertools import chain
from time import sleep

In [3]:
# ==========================
#   Load the scraped names
# ==========================
with open("scraped_names.json", "r", encoding="utf-8") as f:
    scraped_names = json.load(f)

scraped_names = scraped_names
len(scraped_names)

1572

In [4]:
# =====================================
#  Helper Functions for the API Queries
# =====================================

# -- Method for logging -- 
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
LOG_PATH = f"./logs/log_{timestamp}.txt"

def log_line(identifier, logText):
    """
    identifier: something that identifies the logText preferably uniquely for debugging.
    """
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{identifier}\t{logText}\n")
        f.flush()


# -- API backoff --
# Retry on all RequestExceptions, exponential backoff with jitter.
@backoff.on_exception(
    backoff.expo,
    requests.exceptions.RequestException,
    max_tries=7,                     # default infinite — better to cap
    max_time=60,                     # stop after 60 sec
)
def get_with_backoff(url, **kwargs):
    resp = requests.get(url, **kwargs)

    if resp.status_code in (429, 500, 502, 503, 504):
        log_line(resp.status_code, resp.text)
        resp.raise_for_status()

    return resp


# -- Method for retrieving the desired author-information from the API query output -- 
def getPersonInfo(p : dict):
    oid  = (p.get("id") or None)
    sid  = oid[-11:] if len(oid) >= 11 else oid

    display_name = p.get("display_name")
    
    works_api_url = p.get("works_api_url")
    
    h_index = (p.get("summary_stats") or {}).get("h_index")
    
    works_count = p.get("works_count")
    
    # Safe mode of last_known_institutions.country_code
    insts = p.get("last_known_institutions") or []
    countries = [i.get("country_code") for i in insts if (i or {}).get("country_code")]
    country_code = Counter(countries).most_common(1)[0][0] if countries else None

    cited_by_count = p.get("cited_by_count")

    return sid, display_name, works_api_url, h_index, works_count, country_code, cited_by_count

# -- Get and save the person with the maximum number of works and citations --
def getMaxPerson(p : dict, maxworks : int, maxcites : int, maxPerson):

    if p["works_count"] > maxworks: # If works_count is greater than the current for the scraped name
        maxPerson = getPersonInfo(p)
        maxworks = p["works_count"]
        # print("New max person1:", maxPerson)

    elif p["works_count"] == maxworks: # If works_count is equal to the current for the scraped name (tie break)
        if p["cited_by_count"] > maxPerson[-1]:
            maxPerson = getPersonInfo(p)
            # print("New max person2:", maxPerson)

    return maxworks, maxcites, maxPerson

# -- Method for making name tokens for recognizing names like "John Smith" and "Smith John" --
def fuzzy_name_match(a, b):
    score = max(
        fuzz.token_sort_ratio(a, b),
        fuzz.token_set_ratio(a, b)
    )
    return score

# -- Method for removing commas, dots and apostrophes etc. in names like "John Smith" and "Smith, John" so they're properly tokenized --
def cleanName(s):
    s = s.lower()
    for ch in [",", ".", "'", '"']:
        s = s.replace(ch, "")
    return " ".join(s.split())


In [5]:
# -- Method handling the API query results ---
def handle_results(response_json, name):
    maxworks = 0
    maxcites = 0
    maxPers = None
    
    results = response_json.get("results") or []

    name_cf = cleanName(name)

    for p in results:
        alt_list = p.get("display_name_alternatives") or []
        alts_cf = [cleanName(alt) for alt in alt_list]
        display_cf = cleanName(p["display_name"])

        # Exact match
        is_exact = display_cf == name_cf or name_cf in alts_cf
        
        if is_exact:
            maxPers = getPersonInfo(p)
            break

        # Check if there's a match when tokenizing the names of an author
        matched_fuzzy = (
            fuzzy_name_match(name_cf, display_cf) or
            any(fuzzy_name_match(name_cf, alt_cf) for alt_cf in alts_cf)
        )

        if matched_fuzzy > 90:
            maxworks, maxcites, maxPers = getMaxPerson(p, maxworks, maxcites, maxPers)
            continue

        # Levenshtein match
        dist = L.distance(name_cf, display_cf)
        norm = dist / max(len(name_cf), len(display_cf))

        if norm <= 0.25:
            maxworks, maxcites, maxPers = getMaxPerson(p, maxworks, maxcites, maxPers)

    # create final person record
    if maxPers is not None:
        return maxPers
    else:
        return [None, name, None, None, None, None, None]

In [6]:
# ====================================
#          API query method 
# ====================================

def getAuthor(authorNames):
    load_dotenv(override=True)
    API_KEY = os.getenv("OPENALEX_API_KEY")

    BASE = "https://api.openalex.org/authors"
    people = {}

    for name in authorNames:

        # ----------------------------
        # 1. CHEAP QUERY (list query) Cost per query: $0.0001
        # ----------------------------
        cheap_params = {
            "filter": f'display_name:"{name}"',
            # "select": "id,display_name,works_api_url,summary_stats,works_count,last_known_institutions,cited_by_count",
            "per-page": 100,
            "api_key": API_KEY
        }

        r = requests.get(BASE, params=cheap_params).json()

        # print(r)

        if r.get("results"):
            people[name] = handle_results(r, name)
            continue

        # ------------------------------------------
        # 2. EXPENSIVE FALLBACK QUERY (search query)   !! -- Cost per query: $0.001
        # ------------------------------------------
        log_line(name, f"Cheap search failed. Trying expensive full search for: {name}")

        expensive_params = {
            "search": name,   # FULL TEXT SEARCH (expensive)
            "select": "id,display_name,works_api_url,summary_stats,works_count,last_known_institutions,cited_by_count",
            "per-page": 200,
            "api_key": API_KEY
        }

        r2 = requests.get(BASE, params=expensive_params).json()

        if r2.get("results"):
            people[name] = handle_results(r2, name)
        else:
            people[name] = [None, name, None, None, None, None, None]

    return people

In [ ]:
# =============================================
# Parallel processing of API queries using tqdm
# =============================================

chunks = [[n] for n in scraped_names]
all_results = thread_map(getAuthor, chunks, max_workers=8, chunksize=1, desc="Fetching") # Used $0.3827 of API queries

merged = {}
for d in all_results:
    merged.update(d)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
df = pd.DataFrame(merged.values(), columns=['id', 'display_name', "works_api_url", "h_index", "works_count", "country_code", "cited_by_count"])
df.to_pickle(f"./datasets/OpenAlexAPI_researchers_{timestamp}.pkl")
df

Fetching: 100%|██████████| 1572/1572 [07:33<00:00,  3.46it/s]


,id,display_name,works_api_url,h_index,works_count,country_code,cited_by_count
0,A5036764962,Miriam Schirmer,https://api.openalex.org/works?filter=author.i...,3.0,14.0,NaN,27.0
1,A5038833789,Julia Mendelsohn,https://api.openalex.org/works?filter=author.i...,9.0,23.0,US,314.0
2,A5091326079,Dustin Wright,https://api.openalex.org/works?filter=author.i...,7.0,26.0,DK,208.0
3,A5064753440,Dietram A. Scheufele,https://api.openalex.org/works?filter=author.i...,76.0,327.0,US,28643.0
4,A5038951661,Ágnes Horvát,https://api.openalex.org/works?filter=author.i...,1.0,5.0,NaN,2.0
...,...,...,...,...,...,...,...
1567,A5122510249,Angélica Sousa da Mata,https://api.openalex.org/works?filter=author.i...,0.0,2.0,NaN,0.0
1568,NaN,Kwabena Afriyie Owusu,NaN,NaN,NaN,NaN,NaN
1569,A5112804589,Denise Pumain,https://api.openalex.org/works?filter=author.i...,3.0,16.0,FR,142.0
1570,A5100624671,Yang Yue,https://api.openalex.org/works?filter=author.i...,35.0,329.0,CN,12145.0


In [7]:
df_name = "OpenAlexAPI_researchers_2026-03-02_16-02-10_final"
df = pd.read_pickle(f"./datasets/{df_name}.pkl")

# pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')

nanids = (df[df["id"].isna()][0:])[["display_name"]]
print(len(nanids))
nanids

90


,display_name
10,Adam (Zhengzi) Zhou
20,Amirhossein Nakhaei
34,IC2S2 Program Chairs
41,Vincent Christoph Brockers
66,Justyna Janczy
...,...
1529,Xinrui Chloe Zhao
1538,Ethel Elikem Mensah
1541,Joaquín Ignacio Villagra Pacheco
1568,Kwabena Afriyie Owusu


### Reflection Questions

**Major issue:** Querying the API for “David Wright” (1st name in the list from the conference) returns 6 people from OpenAlexAPI!
- Solution: decided to connect the scraped names to the OpenAlexAPI name containing the highest “works_count” (i.e. must be a researcher).

**Minor issue:**
The scraped name is exactly “Ágnes Horvát” but that person does not live up to the above rule (solution). 
The top one has the same accents and a high “works_count”, signifying it is most certainly a researcher, but there is a prefix name which we know nothing of.
The 2nd from the top has the exact same spelling (no extra names) but without accents and also a rather high “work_count”. It is ofc. entirely possible that “Ágnes Horvát” in fact just has a work_count of only 5.

A5004566032 Emőke-Ágnes Horvát 72 \
A5058662232 Agnes Horvat 22\
A5099111610 Emőke-Ágnes Horvát 1\
A5110125283 Emőke Ágnes Horvát 1\
A5038951661 Ágnes Horvát 5
-	Solution: consider alternative names:
If display name == scraped name we take it say works_count=8, BUT if someone else with a different display_name has the scraped name as an alternative with a higher works_count say 23, then we select the 23 one?

**Minor issue:** Country code if affiliated to different institutions in different countries?
-	Solution: take the mode of all country codes of recent institutions. If there are no recent institutions then None.


# Part 3: Collect Research Articles

In [12]:
D1_filtered = df[
    df['works_count'].notna() &
    (df['works_count'] >= 5) &
    (df['works_count'] <= 5000)
]
D1_authorIDs = [id for id in D1_filtered["id"]] # Extract IDs as list and remove NaN IDs.

print(len(D1_authorIDs), D1_authorIDs[0:10])

1225 ['A5036764962', 'A5038833789', 'A5091326079', 'A5064753440', 'A5038951661', 'A5069426228', 'A5016322200', 'A5106944544', 'A5027319334', 'A5017109701']


In [13]:
load_dotenv(override=True)
API_KEY = os.getenv("OPENALEX_API_KEY")

BASE = "https://api.openalex.org/works"

def getWorksByAuthorIDs(author_list):
    ids = "|".join(author_list)

    css_ids = "C144024400|C15744967|C162324750|C17744445" # SOC|PSY|ECON|POLSCI
    quant_ids = "C33923547|C121332964|C41008148" # MATH|PHYS|CS

    params = {
            "filter": f"author.id:{ids},cited_by_count:>10,authors_count:<10,concepts.id:{css_ids},concepts.id:{quant_ids}",
            "select": "id,publication_year,cited_by_count,authorships,title,abstract_inverted_index",
            "per_page": 200,
            "api_key": API_KEY,
            "cursor":"*"
        }

    results = []
    max_pages = 50 #safety cap
    pages = 0

    # Loop to query all the pages of works by an author
    while True:

        if pages >= max_pages:
            print(f"Stopped after reaching safety cap of {max_pages} pages.")
            log_line(pages, f"Stopped after reaching safety cap of {max_pages} pages.")
            break

        r = get_with_backoff(BASE, params=params, timeout=60).json()

        page_results = r.get("results")
        if not page_results:
            break  # done
        results.extend(page_results)

        next_cursor = r.get("meta", {}).get("next_cursor")

        if not next_cursor:
            break  # done

        params["cursor"] = next_cursor
        pages += 1
        sleep(0.2) # polite pause; adjust according to hit rate limits
        # print(r["meta"])

    return results


# Parallel processing of API queries using tqdm
chunks = [D1_authorIDs[i:i+25] for i in range(0, len(D1_authorIDs), 25)]
all_results = thread_map(getWorksByAuthorIDs, chunks, max_workers=8, chunksize=1, desc="Fetching")


# ---- Save results as Pandas Dataframes ----
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

flat_results = list(chain.from_iterable(all_results))
df = pd.DataFrame(flat_results)

D2 = df[["id", "publication_year", "cited_by_count", "authorships"]].copy()
D3 = df[["id", "title", "abstract_inverted_index"]].copy()

# Clean IDs
D2["id"] = D2["id"].map(lambda x: x.split("/")[-1] if isinstance(x, str) else x)
D3["id"] = D3["id"].map(lambda x: x.split("/")[-1] if isinstance(x, str) else x)


# Extract author IDs
D2["authors"] = D2["authorships"].apply(
    lambda xs: [] if xs is None else [
        a["author"]["id"].split("/")[-1]
        for a in xs
        if a["author"]["id"] is not None
    ]
)

D2 = D2.drop(columns=["authorships"])

D2.to_pickle(f"./datasets/D2_{timestamp}.pkl")
D3.to_pickle(f"./datasets/D3_{timestamp}.pkl")

D2.head()

Fetching: 100%|██████████| 49/49 [00:40<00:00,  1.20it/s]


,id,publication_year,cited_by_count,authors
0,W2055957857,1999.0,3185,[A5064753440]
1,W2095655043,2013.0,3013,"[A5090665793, A5113226689]"
2,W2002781392,2006.0,2510,"[A5064753440, A5083730471]"
3,W2096974619,2014.0,1830,"[A5082742221, A5113226689, A5002798141, A50314..."
4,W3124726575,2021.0,1016,"[A5020533147, A5076651767, A5052536455, A50215..."


In [14]:
D3.head()

,id,title,abstract_inverted_index
0,W2055957857,Framing as a Theory of Media Effects,"{'Research': [0], 'on': [1], 'framing': [2, 25..."
1,W2095655043,Text as Data: The Promise and Pitfalls of Auto...,"{'Politics': [0], 'and': [1, 9, 67, 96, 99, 10..."
2,W2002781392,"Framing, Agenda Setting, and Priming: The Evol...","{'This': [0], 'special': [1], 'issue': [2], 'o..."
3,W2096974619,Structural Topic Models for Open‐Ended Survey ...,"{'Collection': [0], 'and': [1, 14, 37, 78, 97,..."
4,W3124726575,Shifting attention to accuracy can reduce misi...,None


### Dataset Summary:

In [21]:
print("The dataset D2 contains", len(D2["id"].drop_duplicates()), "unique works.")
unique_ids = set().union(*D2['authors']) # Unite the ID lists into a SET (set items are always unique).
print("Number of unique researchers co-authoring the papers in D2 (IC2S2 papers):", len(unique_ids))

The dataset D2 contains 12317 unique works.
Number of unique researchers co-authoring the papers in D2 (IC2S2 papers): 19487


**Efficiency in code: strategies implemented to make the code more efficient:**

Besides *filtering* the queries according to the specifications set in the assignment - only IC2S2 authors with work count between 5 and 5.000, works having *more than* 10 citations, and *fewer than* 10 authors, and including only topics relevant to Computational Social Science - to increase efficiency only the desired attributes are *selected* to be retrieved in the query: *id, publication_year, cited_by_count, auhtor_ids, title, abstract_inverted_index*.

Furthermore, instead of retrieving the works of a single author ID per query, a *bulk request* of 25 authers are included in each query and *cursor paging*, with 200 works per page, is implemented to handle the handle the load of works returned.

Finally, a tqdm thread_map is implemented to allow *parallel processing* of the requests, substantially increasing efficiency.